In [1]:
# A VaR model is only useful if it's accurate. Backtesting
# is how you statistically prove (or disprove) that.
#
# Two questions to answer:
#   1. KUPIEC: Is the FREQUENCY of exceptions correct?
#              e.g. at 95% VaR, do we breach ~5% of days?
#   2. CHRISTOFFERSEN: Are exceptions INDEPENDENT over time?
#              e.g. do bad days cluster together? (They shouldn't
#              if the model is capturing time-varying volatility)
#
# These are exactly the tests Basel III requires banks to run.

In [2]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import chi2, norm

sns.set_theme(style="darkgrid")

with open("portfolio_data.pkl", "rb") as f:
    data = pickle.load(f)

portfolio_returns = data["portfolio_returns"]
log_returns       = data["log_returns"]
weights           = data["weights"]
cov_matrix        = data["cov_matrix"]
conf_levels       = data["conf_levels"]
lookback          = data["lookback"]   # 252

returns = portfolio_returns.values
dates   = portfolio_returns.index

print("Loaded successfully.")
print(f"Total observations : {len(returns)}")
print(f"Lookback window    : {lookback} days")
print(f"Rolling periods    : {len(returns) - lookback}")

Loaded successfully.
Total observations : 2514
Lookback window    : 252 days
Rolling periods    : 2262


In [ ]:
import os

_cwd = os.getcwd()
_proj_root = _cwd if os.path.exists(os.path.join(_cwd, "README.md")) else os.path.dirname(_cwd)
RESULTS = os.path.join(_proj_root, "results", "phase_4_backtesting")
os.makedirs(RESULTS, exist_ok=True)
print(f"Results folder: {RESULTS}")

In [3]:
# Point-in-time VaR (Phase 2) used ALL history.
# For backtesting we need ROLLING VaR:
#   → On each day t, estimate VaR using only the PRIOR 252 days
#   → Then check: did day t's actual return breach that VaR?
#
# This is the realistic out-of-sample setup regulators require.
# We implement rolling for all three methods.

def rolling_var_hs(returns: np.ndarray, lookback: int, confidence: float) -> np.ndarray:
    """Rolling Historical Simulation VaR."""
    n      = len(returns)
    var    = np.full(n, np.nan)
    for t in range(lookback, n):
        window  = returns[t - lookback : t]
        var[t]  = -np.percentile(window, (1 - confidence) * 100)
    return var


def rolling_var_parametric(
    returns: np.ndarray, lookback: int, confidence: float
) -> np.ndarray:
    """Rolling Parametric VaR (mu=0, rolling sigma)."""
    n   = len(returns)
    var = np.full(n, np.nan)
    z   = norm.ppf(1 - confidence)
    for t in range(lookback, n):
        window  = returns[t - lookback : t]
        sigma   = window.std()
        var[t]  = -(0 + z * sigma)
    return var


def rolling_var_mc(
    returns: np.ndarray,
    log_returns_df: pd.DataFrame,
    weights: np.ndarray,
    lookback: int,
    confidence: float,
    n_sims: int = 5_000,     # reduced for speed in rolling context
) -> np.ndarray:
    """Rolling Monte Carlo VaR with rolling covariance."""
    n   = len(returns)
    var = np.full(n, np.nan)
    np.random.seed(42)
    for t in range(lookback, n):
        window_df  = log_returns_df.iloc[t - lookback : t]
        cov_daily  = window_df.cov().values
        try:
            L = np.linalg.cholesky(cov_daily)
        except np.linalg.LinAlgError:
            # Fallback: use parametric if Cholesky fails
            sigma  = returns[t - lookback : t].std()
            var[t] = -norm.ppf(1 - confidence) * sigma
            continue
        Z              = np.random.standard_normal((n_sims, len(weights)))
        Z_corr         = Z @ L.T
        sim_rets       = Z_corr @ weights
        var[t]         = -np.percentile(sim_rets, (1 - confidence) * 100)
    return var


print("Rolling VaR functions defined.")
print("Computing rolling VaR for all three methods — MC may take ~30s ...")

c = 0.99   # focus on 99% first (regulatory standard)

roll_hs    = rolling_var_hs(returns, lookback, c)
roll_param = rolling_var_parametric(returns, lookback, c)
roll_mc    = rolling_var_mc(returns, log_returns, weights, lookback, c)

print("Done.")

Rolling VaR functions defined.
Computing rolling VaR for all three methods — MC may take ~30s ...
Done.


In [4]:
# Exception (violation) on day t:
#   actual_return[t] < -VaR[t]
#   i.e. the loss exceeded the predicted threshold
#
# exceptions[t] = 1 if violation, 0 otherwise

def get_exceptions(returns: np.ndarray, rolling_var: np.ndarray) -> np.ndarray:
    """
    Returns binary array: 1 = VaR breach on that day, 0 = no breach.
    Only valid from index `lookback` onward (rolling window warmup).
    """
    exc = np.zeros(len(returns), dtype=int)
    valid = ~np.isnan(rolling_var)
    exc[valid] = (returns[valid] < -rolling_var[valid]).astype(int)
    return exc


exc_hs    = get_exceptions(returns, roll_hs)
exc_param = get_exceptions(returns, roll_param)
exc_mc    = get_exceptions(returns, roll_mc)

# Only count valid (post-warmup) period
T = len(returns) - lookback

for name, exc in [("HS", exc_hs), ("Parametric", exc_param), ("MC", exc_mc)]:
    n_exc       = exc[lookback:].sum()
    actual_rate = n_exc / T
    expected    = (1 - c) * T
    print(f"{name:12s} | Exceptions: {n_exc:3d} / {T}"
          f" | Actual rate: {actual_rate:.3%}"
          f" | Expected: {expected:.1f} ({(1-c):.1%})")

HS           | Exceptions:  35 / 2262 | Actual rate: 1.547% | Expected: 22.6 (1.0%)
Parametric   | Exceptions:  50 / 2262 | Actual rate: 2.210% | Expected: 22.6 (1.0%)
MC           | Exceptions:  49 / 2262 | Actual rate: 2.166% | Expected: 22.6 (1.0%)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
valid_dates = dates[lookback:]

configs = [
    ("Historical Simulation", roll_hs,    exc_hs,    "steelblue"),
    ("Parametric",            roll_param, exc_param, "tomato"),
    ("Monte Carlo",           roll_mc,    exc_mc,    "mediumpurple"),
]

for ax, (name, roll_var, exc, color) in zip(axes, configs):
    actual = returns[lookback:] * 100
    var_   = roll_var[lookback:] * 100
    exc_   = exc[lookback:]

    ax.plot(valid_dates, actual, color="grey", linewidth=0.6,
            alpha=0.7, label="Actual return")
    ax.plot(valid_dates, -var_, color=color, linewidth=1.2,
            label=f"{int(c*100)}% VaR threshold")
    ax.scatter(valid_dates[exc_ == 1], actual[exc_ == 1],
               color="red", s=15, zorder=5, label="Exception")

    ax.set_title(f"{name} — Rolling {int(c*100)}% VaR vs Actual Returns")
    ax.set_ylabel("Return (%)")
    ax.legend(fontsize=8, loc="lower right")

axes[-1].set_xlabel("Date")
plt.suptitle("Rolling VaR Backtesting — Exception Identification",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "01_rolling_var_vs_returns.png"), dpi=150, bbox_inches="tight")
plt.show()

In [6]:
#Tests: is the PROPORTION of exceptions statistically consistent
#        with the model's confidence level?
#
# H0: p_actual = p_expected = (1 - c)
# H1: p_actual ≠ p_expected
#
# Likelihood Ratio statistic:
#   LR_uc = -2 * ln[(1-p*)^(T-N) * p*^N]
#           +2 * ln[(1-N/T)^(T-N) * (N/T)^N]
#
# where:
#   T  = total observations in rolling window period
#   N  = number of exceptions
#   p* = expected exception rate = (1 - c)
#
# LR_uc ~ chi2(df=1) under H0
# Reject H0 (model is BAD) if LR_uc > chi2 critical value

def kupiec_test(exceptions: np.ndarray, confidence: float, lookback: int) -> dict:
    """
    Kupiec Proportion of Failures (POF) test.

    Returns dict with:
        N        : number of exceptions
        T        : total observations
        p_hat    : actual exception rate
        p_star   : expected exception rate
        LR_uc    : likelihood ratio statistic
        p_value  : p-value against chi2(1)
        reject   : True if model fails at 5% significance
    """
    exc    = exceptions[lookback:]
    T      = len(exc)
    N      = exc.sum()
    p_hat  = N / T               # actual rate
    p_star = 1 - confidence      # expected rate

    # Avoid log(0) edge cases
    if N == 0 or N == T:
        return {"N": N, "T": T, "p_hat": p_hat, "p_star": p_star,
                "LR_uc": np.nan, "p_value": np.nan, "reject": True}

    LR_uc = -2 * (
        np.log((1 - p_star) ** (T - N) * p_star ** N)
        - np.log((1 - p_hat) ** (T - N) * p_hat ** N)
    )

    p_value = 1 - chi2.cdf(LR_uc, df=1)
    reject  = p_value < 0.05    # 5% significance level

    return {"N": N, "T": T, "p_hat": p_hat, "p_star": p_star,
            "LR_uc": LR_uc, "p_value": p_value, "reject": reject}


print(f"── Kupiec Test Results ({int(c*100)}% VaR) ────────────────────────")
print(f"{'Method':15s} {'N':>5s} {'T':>6s} {'p_hat':>8s} "
      f"{'p_expected':>10s} {'LR_uc':>8s} {'p-value':>9s} {'Result':>8s}")
print("─" * 75)

kupiec_results = {}
for name, exc in [("HS", exc_hs), ("Parametric", exc_param), ("MC", exc_mc)]:
    r = kupiec_test(exc, c, lookback)
    kupiec_results[name] = r
    result = "FAIL ✗" if r["reject"] else "PASS ✓"
    print(f"{name:15s} {r['N']:>5d} {r['T']:>6d} {r['p_hat']:>8.3%} "
          f"{r['p_star']:>10.3%} {r['LR_uc']:>8.3f} {r['p_value']:>9.4f} {result:>8s}")

# Interpretation:
# PASS = exception rate is statistically consistent with model's confidence level
# FAIL = too many or too few exceptions → model is miscalibrated

── Kupiec Test Results (99% VaR) ────────────────────────
Method              N      T    p_hat p_expected    LR_uc   p-value   Result
───────────────────────────────────────────────────────────────────────────
HS                 35   2262   1.547%     1.000%    5.865    0.0154   FAIL ✗
Parametric         50   2262   2.210%     1.000%   24.895    0.0000   FAIL ✗
MC                 49   2262   2.166%     1.000%   23.305    0.0000   FAIL ✗


In [7]:
# Even if frequency is right (Kupiec passes), exceptions might CLUSTER in time.
# Clustering means the model fails to capture volatility dynamics —
# a bad day predicts another bad day, which the model misses.
#
# Christoffersen tests INDEPENDENCE of exceptions using a transition matrix:
#
#   n00 = days with no exception FOLLOWING a no-exception day
#   n01 = days with exception FOLLOWING a no-exception day
#   n10 = days with no exception FOLLOWING an exception day
#   n11 = days with exception FOLLOWING an exception day
#
# If exceptions are independent:  π01 = π11  (no "memory")
# If clustered:                   π11 > π01  (exception predicts exception)
#
# LR_ind ~ chi2(df=1)
# Full Christoffersen LR_cc = LR_uc + LR_ind ~ chi2(df=2)

def christoffersen_test(exceptions: np.ndarray, confidence: float, lookback: int) -> dict:
    """
    Christoffersen Conditional Coverage test.

    Returns dict with full test components:
        n00, n01, n10, n11  : transition counts
        pi01, pi11          : transition probabilities
        LR_ind              : independence LR stat ~ chi2(1)
        LR_cc               : full conditional coverage = LR_uc + LR_ind ~ chi2(2)
        p_value_ind         : p-value for independence test
        p_value_cc          : p-value for full conditional coverage test
        reject_ind          : independence test result
        reject_cc           : conditional coverage test result
    """
    exc = exceptions[lookback:].astype(int)
    T   = len(exc)

    # Build transition counts
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        prev, curr = exc[t-1], exc[t]
        if   prev == 0 and curr == 0: n00 += 1
        elif prev == 0 and curr == 1: n01 += 1
        elif prev == 1 and curr == 0: n10 += 1
        elif prev == 1 and curr == 1: n11 += 1

    # Transition probabilities
    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0  # P(exception | no prior exception)
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0  # P(exception | prior exception)
    pi   = (n01 + n11) / T                               # unconditional exception rate

    # Independence LR statistic
    # Avoids log(0)
    eps = 1e-10
    LR_ind = -2 * (
        np.log(pi + eps) * (n01 + n11)
        + np.log(1 - pi + eps) * (n00 + n10)
        - np.log(pi01 + eps) * n01
        - np.log(1 - pi01 + eps) * n00
        - np.log(pi11 + eps) * n11
        - np.log(1 - pi11 + eps) * n10
    )
    LR_ind = max(LR_ind, 0)   # numerical safety

    # Full conditional coverage = Kupiec + Independence
    kupiec = kupiec_test(exceptions, confidence, lookback)
    LR_uc  = kupiec["LR_uc"] if not np.isnan(kupiec["LR_uc"]) else 0
    LR_cc  = LR_uc + LR_ind

    p_value_ind = 1 - chi2.cdf(LR_ind, df=1)
    p_value_cc  = 1 - chi2.cdf(LR_cc,  df=2)

    return {
        "n00": n00, "n01": n01, "n10": n10, "n11": n11,
        "pi01": pi01, "pi11": pi11,
        "LR_ind": LR_ind, "LR_cc": LR_cc,
        "p_value_ind": p_value_ind, "p_value_cc": p_value_cc,
        "reject_ind": p_value_ind < 0.05,
        "reject_cc":  p_value_cc  < 0.05,
    }


print(f"── Christoffersen Test Results ({int(c*100)}% VaR) ──────────────────")

christoffersen_results = {}
for name, exc in [("HS", exc_hs), ("Parametric", exc_param), ("MC", exc_mc)]:
    r = christoffersen_test(exc, c, lookback)
    christoffersen_results[name] = r

    print(f"\n  {name}")
    print(f"    Transition counts  : n00={r['n00']:4d}  n01={r['n01']:4d}"
          f"  n10={r['n10']:4d}  n11={r['n11']:4d}")
    print(f"    π01 (exc | no exc) : {r['pi01']:.4f}")
    print(f"    π11 (exc | exc)    : {r['pi11']:.4f}  ← should ≈ π01 if independent")
    print(f"    LR_ind             : {r['LR_ind']:.4f}  p={r['p_value_ind']:.4f}"
          f"  {'FAIL ✗' if r['reject_ind'] else 'PASS ✓'}")
    print(f"    LR_cc (full test)  : {r['LR_cc']:.4f}   p={r['p_value_cc']:.4f}"
          f"  {'FAIL ✗' if r['reject_cc']  else 'PASS ✓'}")

── Christoffersen Test Results (99% VaR) ──────────────────

  HS
    Transition counts  : n00=2196  n01=  30  n10=  30  n11=   5
    π01 (exc | no exc) : 0.0135
    π11 (exc | exc)    : 0.1429  ← should ≈ π01 if independent
    LR_ind             : 14.5226  p=0.0001  FAIL ✗
    LR_cc (full test)  : 20.3871   p=0.0000  FAIL ✗

  Parametric
    Transition counts  : n00=2167  n01=  44  n10=  44  n11=   6
    π01 (exc | no exc) : 0.0199
    π11 (exc | exc)    : 0.1200  ← should ≈ π01 if independent
    LR_ind             : 11.5321  p=0.0007  FAIL ✗
    LR_cc (full test)  : 36.4271   p=0.0000  FAIL ✗

  MC
    Transition counts  : n00=2167  n01=  45  n10=  45  n11=   4
    π01 (exc | no exc) : 0.0203
    π11 (exc | exc)    : 0.0816  ← should ≈ π01 if independent
    LR_ind             : 5.1053  p=0.0239  FAIL ✗
    LR_cc (full test)  : 28.4099   p=0.0000  FAIL ✗


In [8]:
rows = []
for name in ["HS", "Parametric", "MC"]:
    k = kupiec_results[name]
    cr = christoffersen_results[name]
    rows.append({
        "Method":           name,
        "Exceptions (N)":   k["N"],
        "Expected":         round((1 - c) * k["T"], 1),
        "Act. Rate":        f"{k['p_hat']:.3%}",
        "Kupiec p-val":     round(k["p_value"], 4),
        "Kupiec":           "PASS ✓" if not k["reject"] else "FAIL ✗",
        "π01":              round(cr["pi01"], 4),
        "π11":              round(cr["pi11"], 4),
        "CC p-val":         round(cr["p_value_cc"], 4),
        "Christoffersen":   "PASS ✓" if not cr["reject_cc"] else "FAIL ✗",
    })

summary = pd.DataFrame(rows).set_index("Method")
print(f"\n── Combined Backtesting Summary ({int(c*100)}% VaR) ──────────────")
summary


── Combined Backtesting Summary (99% VaR) ──────────────


,Exceptions (N),Expected,Act. Rate,Kupiec p-val,Kupiec,π01,π11,CC p-val,Christoffersen
Method,,,,,,,,,
HS,35,22.6,1.547%,0.0154,FAIL ✗,0.0135,0.1429,0.0,FAIL ✗
Parametric,50,22.6,2.210%,0.0000,FAIL ✗,0.0199,0.1200,0.0,FAIL ✗
MC,49,22.6,2.166%,0.0000,FAIL ✗,0.0203,0.0816,0.0,FAIL ✗


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 7), sharex=True)
valid_dates = dates[lookback:]

for ax, (name, key, exc, color) in zip(axes, [
    ("Historical Sim", "HS",          exc_hs,    "steelblue"),
    ("Parametric",     "Parametric",  exc_param, "tomato"),
    ("Monte Carlo",    "MC",          exc_mc,    "mediumpurple"),
]):
    exc_ = exc[lookback:]
    exc_dates = valid_dates[exc_ == 1]
    ax.vlines(exc_dates, 0, 1, color=color, alpha=0.7, linewidth=0.8)
    ax.set_yticks([])
    ax.set_ylabel(name, fontsize=9)
    n    = exc_.sum()
    pi11 = christoffersen_results[key]["pi11"]
    ax.set_title(f"{name} — {n} exceptions  |  π11={pi11:.3f}", fontsize=9)

axes[-1].set_xlabel("Date")
plt.suptitle(f"Exception Timeline — {int(c*100)}% VaR  (vertical lines = breach days)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "02_exception_timeline.png"), dpi=150, bbox_inches="tight")
plt.show()

In [10]:
print("── Full Backtesting Results: 95% and 99% VaR ───────────\n")

all_results = {}
for conf in conf_levels:
    rv_hs    = rolling_var_hs(returns, lookback, conf)
    rv_param = rolling_var_parametric(returns, lookback, conf)
    rv_mc    = rolling_var_mc(returns, log_returns, weights, lookback, conf)

    e_hs    = get_exceptions(returns, rv_hs)
    e_param = get_exceptions(returns, rv_param)
    e_mc    = get_exceptions(returns, rv_mc)

    print(f"  ── {int(conf*100)}% VaR ──────────────────────────────────────────")
    print(f"  {'Method':12s} {'N':>4s} {'Rate':>7s} {'Kupiec':>9s} {'CC Test':>9s}")
    for name, exc in [("HS", e_hs), ("Parametric", e_param), ("MC", e_mc)]:
        k  = kupiec_test(exc, conf, lookback)
        cr = christoffersen_test(exc, conf, lookback)
        print(f"  {name:12s} {k['N']:>4d} {k['p_hat']:>7.3%}"
              f"  {'PASS ✓' if not k['reject']    else 'FAIL ✗':>9s}"
              f"  {'PASS ✓' if not cr['reject_cc'] else 'FAIL ✗':>9s}")

    all_results[conf] = {
        "roll_hs": rv_hs, "roll_param": rv_param, "roll_mc": rv_mc,
        "exc_hs":  e_hs,  "exc_param":  e_param,  "exc_mc":  e_mc,
    }
    print()

── Full Backtesting Results: 95% and 99% VaR ───────────

  ── 95% VaR ──────────────────────────────────────────
  Method          N    Rate    Kupiec   CC Test
  HS            123  5.438%     PASS ✓     FAIL ✗
  Parametric    106  4.686%     PASS ✓     FAIL ✗
  MC            105  4.642%     PASS ✓     FAIL ✗

  ── 99% VaR ──────────────────────────────────────────
  Method          N    Rate    Kupiec   CC Test
  HS             35  1.547%     FAIL ✗     FAIL ✗
  Parametric     50  2.210%     FAIL ✗     FAIL ✗
  MC             49  2.166%     FAIL ✗     FAIL ✗



In [11]:
backtest_results = {
    "backtest_all_results": all_results,
    "kupiec_results_99":    kupiec_results,
    "christoffersen_99":    christoffersen_results,
    "exc_hs_99":            exc_hs,
    "exc_param_99":         exc_param,
    "exc_mc_99":            exc_mc,
    "roll_hs_99":           roll_hs,
    "roll_param_99":        roll_param,
    "roll_mc_99":           roll_mc,
}

data.update(backtest_results)

with open("portfolio_data.pkl", "wb") as f:
    pickle.dump(data, f)

print("portfolio_data.pkl updated with Phase 4 backtesting results.")

portfolio_data.pkl updated with Phase 4 backtesting results.
